# Step 2 - Exploratory Data Analysis

This notebook explores the cleaned PJM East hourly demand series and documents the patterns that will guide the later stationarity, seasonality, and SARIMA modeling steps.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

PROCESSED_PATH = "../data/processed/"
FIGURES_PATH = "../reports/figures/"

os.makedirs(FIGURES_PATH, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)

df = pd.read_csv(
    os.path.join(PROCESSED_PATH, "pjme_clean.csv"),
    parse_dates=["Datetime"],
    index_col="Datetime",
).sort_index()

series = df["PJME_MW"].astype(float)
df["PJME_MW"] = series

print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Missing values: {series.isna().sum()}")

df.head()

In [ ]:
summary_stats = pd.DataFrame(
    {
        "metric": [
            "mean_mw",
            "std_mw",
            "min_mw",
            "max_mw",
            "skewness",
            "kurtosis",
            "variance_mw",
            "range_mw",
        ],
        "value": [
            series.mean(),
            series.std(),
            series.min(),
            series.max(),
            series.skew(),
            series.kurt(),
            series.var(),
            series.max() - series.min(),
        ],
    }
)

print("Descriptive statistics for PJME_MW")
display(summary_stats.style.format({"value": "{:,.4f}"}))

The average hourly demand is about 32,080 MW, but the spread is wide with a standard deviation above 6,400 MW and a range of 47,465 MW. Positive skewness and positive kurtosis already suggest that the distribution is not perfectly symmetric, so later modeling steps should not assume normality without checking.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=False)

series.plot(ax=axes[0], linewidth=0.5, color="steelblue")
axes[0].set_title("Full series: 2002 to 2018")
axes[0].set_ylabel("Demand (MW)")

df.loc["2017", "PJME_MW"].plot(ax=axes[1], linewidth=0.8, color="steelblue")
axes[1].set_title("Zoomed view: 2017")
axes[1].set_ylabel("Demand (MW)")

df.loc["2017-01", "PJME_MW"].plot(ax=axes[2], linewidth=1.2, color="steelblue")
axes[2].set_title("Zoomed view: January 2017")
axes[2].set_ylabel("Demand (MW)")
axes[2].set_xlabel("Datetime")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, "02_line_plots.png"), dpi=150, bbox_inches="tight")
plt.show()

The full series shows strong repeating peaks and troughs across years, while the 2017 and January 2017 zooms make the shorter daily and weekly cycles much clearer. This means the series is highly seasonal at multiple horizons, so the next stages should explicitly test for seasonality and likely favor SARIMA over a simple non-seasonal ARIMA model.

In [ ]:
df_scatter = df.assign(hour=df.index.hour, month=df.index.month)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

axes[0].scatter(df_scatter["hour"], df_scatter["PJME_MW"], alpha=0.01, s=2, color="steelblue")
axes[0].set_title("Hour of day vs demand")
axes[0].set_xlabel("Hour of day")
axes[0].set_ylabel("Demand (MW)")
axes[0].set_xticks(range(24))

axes[1].scatter(df_scatter["month"], df_scatter["PJME_MW"], alpha=0.01, s=2, color="coral")
axes[1].set_title("Month vs demand")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Demand (MW)")
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, "02_scatter_plots.png"), dpi=150, bbox_inches="tight")
plt.show()

The hour-level scatter shows that demand is not randomly spread through the day; some hours consistently cluster at higher loads than others. The month-level scatter also shows broader seasonal variation, with colder and hotter parts of the year reaching higher demand, so both intra-day and annual effects should be treated as real structure rather than noise.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))

sns.histplot(series, bins=80, kde=True, color="steelblue", edgecolor="white", ax=ax)
ax.set_title("Distribution of PJME hourly demand")
ax.set_xlabel("Demand (MW)")
ax.set_ylabel("Frequency")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, "02_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

The histogram and KDE indicate a slightly right-skewed distribution rather than a symmetric bell shape. That matters because it hints that transformations such as log or differencing may be worth testing later if we need to stabilize variance or improve residual behavior.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

stats.probplot(series.dropna(), dist="norm", plot=ax)
ax.set_title("QQ plot: PJME demand vs normal distribution")
ax.get_lines()[0].set(markersize=1.2, alpha=0.35, color="steelblue")
ax.get_lines()[1].set(color="firebrick", linewidth=1.5)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, "02_qq_plot.png"), dpi=150, bbox_inches="tight")
plt.show()

If the demand values were normally distributed, most points would lie close to the straight reference line. Instead, the tails bend away from that line, which confirms heavier tails and mild skewness; this supports using the QQ plot as evidence that normality is imperfect and should not drive model choice.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 11))

sns.boxplot(data=df_scatter, x="hour", y="PJME_MW", ax=axes[0], color="lightsteelblue", fliersize=0.7)
axes[0].set_title("Demand by hour of day")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Demand (MW)")

sns.boxplot(data=df_scatter, x="month", y="PJME_MW", ax=axes[1], color="lightsalmon", fliersize=0.7)
axes[1].set_title("Demand by month of year")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Demand (MW)")
axes[1].set_xticks(range(12))
axes[1].set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, "02_boxplots.png"), dpi=150, bbox_inches="tight")
plt.show()

The boxplots make the seasonal structure easier to compare by median and spread. Demand changes meaningfully across hours and months, which reinforces that the series contains systematic daily and annual patterns and gives a strong reason to test seasonal decomposition and seasonal differencing in later steps.

In [ ]:
eda_summary = pd.DataFrame(
    [
        {
            "n_observations": len(df),
            "start_datetime": df.index.min(),
            "end_datetime": df.index.max(),
            "mean_mw": series.mean(),
            "std_mw": series.std(),
            "min_mw": series.min(),
            "max_mw": series.max(),
            "skewness": series.skew(),
            "kurtosis": series.kurt(),
            "variance_mw": series.var(),
            "range_mw": series.max() - series.min(),
        }
    ]
)

eda_summary.to_csv(os.path.join(PROCESSED_PATH, "eda_summary.csv"), index=False)
print("EDA summary saved to ../data/processed/eda_summary.csv")
eda_summary